ResNet 残差神经网络
提出residual结构（残差结构），并搭建超深的网络结构(突破1000层)
使用Batch Normalization加速训练(丢弃dropout)

层数较少
两层3*3，主分支和捷径分支，shape要一致
主分支（Main Branch）：经过卷积、归一化等操作，学习需要增加或调整的特征。
捷径分支（Shortcut）：让输入绕过主分支，直接传递到相加的位置。

In [1]:
import torch
import torch.nn as nn

In [ ]:
x = torch.randn(2,3,32,32)

conv1 = nn.Conv2d(3,3,kernel_size=3,padding=1)#输出可以自己设计，最后残差出来的shape，与跳跃连接出来shape一致就行（卷积投影等方式调整跳跃连接的尺寸。）
conv2 = nn.Conv2d(3,3,kernel_size=3,padding=1)

residual = conv2(torch.relu(conv1(x)))

y = residual + x  #形状必须相同

print("输入形状",x.shape)
print("残差shape",residual.shape)
print("输出形状",y.shape)

输入形状 torch.Size([2, 3, 32, 32])
残差shape torch.Size([2, 3, 32, 32])
输出形状 torch.Size([2, 3, 32, 32])


不改变shape版，ResidualBlock

In [ ]:
class My_ResNet(nn.Module):
    def __init__(self,channels):
        super().__init__()

        self.conv1 = nn.Conv2d(channels,channels,kernel_size=3,stride=1,padding=1,bias=False)
        self.bn1 = nn.BatchNorm2d(channels)#通道维度，统计量归一化
        self.conv2 = nn.Conv2d(channels,channels,kernel_size=3,stride=1,padding=1,bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU(inplace=True)#就地修改，省内存，但是有风险

    def forward(self,x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out = out + identity
        out = self.relu(out)

        return out

if __name__ =="__main__":
    model = My_ResNet(channels=64)

    x = torch.randn(4,64,32,32)

    y = model(x)

    print("out.shape:",y.shape)
    print("参数量",sum(p.numel() for p in model.parameters()))


out.shape: torch.Size([4, 64, 32, 32])
参数量 74112


Conv 后面紧跟 BatchNorm	bias=False ✅（bias 无意义）
Conv 后面没有 BN（直接接 ReLU 或输出）	bias=True ✅（需要偏置来平移激活）

In [37]:
class residualBlock(nn.Module):
    def __init__(self,in_channels,out_channels,stride = 1):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels,out_channels,kernel_size=3,stride=stride,padding=1,bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels,out_channels,kernel_size=3,stride=1,padding=1,bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.shortcut = nn.Identity()
        #shape不等使用1*1卷积核
        if stride!=1 or in_channels!=out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels,out_channels,kernel_size=1,stride=stride,bias=False
                ),
                nn.BatchNorm2d(out_channels)
            )
        self.fla = nn.Flatten()#全连接之前要展平
        self.linear = nn.Linear(out_channels*16*16,4)
    
    def forward(self, x):
        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = out + identity
        out = self.relu(out)
        out = self.fla(out)
        out = self.linear(out)

        return out


if __name__ == "__main__":
    """ # 情况一：尺寸不变
    block1 = residualBlock(64, 64)
    x1 = torch.randn(4, 64, 32, 32)
    print("情况一：", block1(x1).shape)

    # 情况二：通道数和空间大小都改变
    block2 = residualBlock(64, 128, stride=2)
    x2 = torch.randn(4, 64, 32, 32)
    print("情况二：", block2(x2).shape)
"""
    x3 = torch.randn(4, 64, 32, 32)
    y = torch.randint(0,4,(4,))
    print("y_value",y)
    block3 = residualBlock(64, 256, stride=2)
    y_pred = block3(x3)
    loss = nn.CrossEntropyLoss()
    loss_value = loss(y_pred,y)
    loss_value.backward()
    logits = y_pred.argmax(dim=1)
    print("loss_value",loss_value)
    print(y_pred)
    print("y_pred.shape",y_pred.shape)
    print("预测值",logits)


y_value tensor([1, 2, 2, 1])
loss_value tensor(1.5509, grad_fn=<NllLossBackward0>)
tensor([[ 0.2209,  0.3144, -0.0441,  0.0632],
        [ 0.4665, -0.0284,  0.5253, -1.1457],
        [ 0.5339,  0.6824, -0.3212,  0.1385],
        [-0.1513, -0.6548, -0.1317,  0.3379]], grad_fn=<AddmmBackward0>)
y_pred.shape torch.Size([4, 4])
预测值 tensor([1, 2, 1, 3])
